In [2]:
import os

folder = "train"  # adjust if your folder is named differently

files = ["train_source1.tsv", "train_source2.tsv", "train_source3.tsv", "train_ground_truth.tsv"]

for f in files:
    path = os.path.join(folder, f)
    with open(path, "r", encoding="utf-8") as file:
        count = sum(1 for _ in file)
    print(f, count)

train_source1.tsv 2206822
train_source2.tsv 5034617
train_source3.tsv 5285604
train_ground_truth.tsv 2206822


In [5]:
import pandas as pd

gt = pd.read_csv("train/train_ground_truth.tsv", sep="\t")
print(gt.shape)
gt.head()

(2206821, 2)


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [6]:
gt['is_singleton'] = gt['matched_entity_ids'].isna()
print("Singleton rate:", gt['is_singleton'].mean())

match_counts = gt['matched_entity_ids'].dropna().str.split(',').str.len()
print(match_counts.describe())
print(match_counts.value_counts().sort_index().head(20))

Singleton rate: 0.05584820880352326
count    2.083574e+06
mean     3.665992e+00
std      1.526294e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: matched_entity_ids, dtype: float64
matched_entity_ids
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64


In [7]:
s1 = pd.read_csv("train/train_source1.tsv", sep="\t")

# Take a few real matched rows
sample_matches = gt[~gt['is_singleton']].sample(8, random_state=1)

for _, row in sample_matches.iterrows():
    s1_id = row['source1_entity_id']
    s1_rec = s1[s1['entity_id'] == s1_id]
    print("=== S1:", s1_id, "===")
    print(s1_rec[['business_name', 'business_address', 'country']].to_string(index=False))
    print("Matches:", row['matched_entity_ids'])
    print()

=== S1: S1-709830881 ===
               business_name                                                          business_address country
Bnp Group of Companies-Delhi House No.-591, G.F, Bhai Parmanand Colony, Delhi, North West Delhi, Delhi   India
Matches: S2-669695025,S3-346267754,S3-154538164

=== S1: S1-552405232 ===
    business_name                                                                             business_address country
Svl Tubes Pvt Ltd Plot No: 51 First Floor, Govindan Street, Ayyavoo Colony, Aminjik, Arai, Chennai, Tamil Nadu   India
Matches: S2-427072237,S2-194485792,S3-173273014,S3-935107115,S3-255079928

=== S1: S1-575257338 ===
  business_name            business_address country
Jex First Focus 920 Stinocher St, Solon, IA      US
Matches: S2-210742328,S2-178878186,S3-197865866,S3-976385621,S3-234766100

=== S1: S1-546069131 ===
                  business_name                                             business_address country
Tirth Resources Private Limited Jaip

In [8]:
s2 = pd.read_csv("train/train_source2.tsv", sep="\t")
s3 = pd.read_csv("train/train_source3.tsv", sep="\t")

print(s2.shape, s3.shape)

(5034616, 4) (5285603, 4)


In [9]:
def show_match(s1_id):
    rec = s1[s1['entity_id'] == s1_id]
    print("=== S1:", s1_id, "===")
    print(rec[['business_name', 'business_address', 'country']].to_string(index=False))
    
    matched_ids = gt[gt['source1_entity_id'] == s1_id]['matched_entity_ids'].values[0].split(',')
    for mid in matched_ids:
        src = s2 if mid.startswith('S2') else s3
        m = src[src['entity_id'] == mid]
        print(f"--- {mid} ---")
        print(m[['business_name', 'business_address', 'country']].to_string(index=False))
    print()

# Try a few from your sample above
for sid in ["S1-709830881", "S1-552405232", "S1-575257338", "S1-546069131", "S1-372901971"]:
    show_match(sid)

=== S1: S1-709830881 ===
               business_name                                                          business_address country
Bnp Group of Companies-Delhi House No.-591, G.F, Bhai Parmanand Colony, Delhi, North West Delhi, Delhi   India
--- S2-669695025 ---
                       business_name                                                          business_address country
of Group Bnp Companies-Delhi Limited NORTH WEST DELHI, HOUSE NO.-591, Delhi, G.F, BHAI PARMANAND COLONY, DELHI   India
--- S3-346267754 ---
                business_name                                                       business_address country
Bnp  Group-of Companies-Delhi House No.-591, G.f, Bhai Parmanand Colony, Delhi, North West Delhi, DL   India
--- S3-154538164 ---
             business_name                                                       business_address country
bnpgroupcompaniesdelhi.com House No.-591, G.f, Bhai Parmanand Colony, North West Delhi, Delhi, DL   India

=== S1: S1-552405232 

In [10]:
import re

# 1. How many S2/S3 records are domain/handle-style names?
domain_pattern = s2['business_name'].str.contains(r'\.com|\.in|@', regex=True, na=False)
print("S2 domain-style names:", domain_pattern.sum(), f"({domain_pattern.mean():.2%})")

domain_pattern_s3 = s3['business_name'].str.contains(r'\.com|\.in|@', regex=True, na=False)
print("S3 domain-style names:", domain_pattern_s3.sum(), f"({domain_pattern_s3.mean():.2%})")

# 2. How many contain "formerly" / rebranding language?
formerly_s2 = s2['business_name'].str.contains(r'formerly', case=False, na=False)
formerly_s3 = s3['business_name'].str.contains(r'formerly', case=False, na=False)
print("S2 'formerly' mentions:", formerly_s2.sum())
print("S3 'formerly' mentions:", formerly_s3.sum())

# 3. How many contain non-Latin script characters?
non_latin_s2 = s2['business_name'].str.contains(r'[^\x00-\x7F]', regex=True, na=False)
non_latin_s2_addr = s2['business_address'].str.contains(r'[^\x00-\x7F]', regex=True, na=False)
print("S2 non-Latin names:", non_latin_s2.sum(), f"({non_latin_s2.mean():.2%})")
print("S2 non-Latin addresses:", non_latin_s2_addr.sum(), f"({non_latin_s2_addr.mean():.2%})")

non_latin_s3 = s3['business_name'].str.contains(r'[^\x00-\x7F]', regex=True, na=False)
non_latin_s3_addr = s3['business_address'].str.contains(r'[^\x00-\x7F]', regex=True, na=False)
print("S3 non-Latin names:", non_latin_s3.sum(), f"({non_latin_s3.mean():.2%})")
print("S3 non-Latin addresses:", non_latin_s3_addr.sum(), f"({non_latin_s3_addr.mean():.2%})")

S2 domain-style names: 177158 (3.52%)
S3 domain-style names: 185732 (3.51%)
S2 'formerly' mentions: 0
S3 'formerly' mentions: 30057
S2 non-Latin names: 764608 (15.19%)
S2 non-Latin addresses: 478453 (9.50%)
S3 non-Latin names: 606737 (11.48%)
S3 non-Latin addresses: 476588 (9.02%)


In [2]:
import pandas as pd

s1 = pd.read_csv("train/train_source1.tsv", sep="\t")
s2 = pd.read_csv("train/train_source2.tsv", sep="\t")
s3 = pd.read_csv("train/train_source3.tsv", sep="\t")
gt = pd.read_csv("train/train_ground_truth.tsv", sep="\t")

print(s1.shape, s2.shape, s3.shape, gt.shape)

(2206821, 4) (5034616, 4) (5285603, 4) (2206821, 2)


In [3]:
import re
import pandas as pd

def audit_noise(df, name):
    print(f"\n{'='*20} {name} {'='*20}")
    n = len(df)
    
    # Missing data
    print(f"Missing business_name: {df['business_name'].isna().sum()} ({df['business_name'].isna().mean():.2%})")
    print(f"Missing business_address: {df['business_address'].isna().sum()} ({df['business_address'].isna().mean():.2%})")
    
    # Non-Latin script
    non_latin_name = df['business_name'].str.contains(r'[^\x00-\x7F]', regex=True, na=False)
    non_latin_addr = df['business_address'].str.contains(r'[^\x00-\x7F]', regex=True, na=False)
    print(f"Non-Latin names: {non_latin_name.sum()} ({non_latin_name.mean():.2%})")
    print(f"Non-Latin addresses: {non_latin_addr.sum()} ({non_latin_addr.mean():.2%})")
    
    # Domain/handle style
    domain = df['business_name'].str.contains(r'\.(com|in|org|net|co)\b|^@', regex=True, na=False)
    print(f"Domain/handle names: {domain.sum()} ({domain.mean():.2%})")
    
    # "Formerly" rebranding
    formerly = df['business_name'].str.contains(r'formerly', case=False, na=False)
    print(f"'Formerly' mentions: {formerly.sum()} ({formerly.mean():.2%})")
    
    # All-caps (formatting inconsistency signal)
    all_caps = df['business_name'].str.isupper()
    print(f"All-caps names: {all_caps.sum()} ({all_caps.mean():.2%})")
    
    # Legal suffix presence (rough check)
    suffix_pattern = r'\b(ltd|limited|pvt|private|inc|incorporated|corp|corporation|llc|llp|co)\b'
    has_suffix = df['business_name'].str.contains(suffix_pattern, case=False, regex=True, na=False)
    print(f"Has legal suffix: {has_suffix.sum()} ({has_suffix.mean():.2%})")
    
    # Address: PIN/ZIP code presence
    has_digits_addr = df['business_address'].str.contains(r'\d{5,6}', regex=True, na=False)
    print(f"Address has 5-6 digit code (PIN/ZIP): {has_digits_addr.sum()} ({has_digits_addr.mean():.2%})")
    
    # Address length distribution (proxy for partial vs full addresses)
    addr_len = df['business_address'].dropna().str.len()
    print(f"Address length - mean: {addr_len.mean():.1f}, median: {addr_len.median():.0f}, min: {addr_len.min()}, max: {addr_len.max()}")
    
    # Country breakdown
    print(f"Country distribution:\n{df['country'].value_counts()}")

audit_noise(s1, "Source 1")
audit_noise(s2, "Source 2")
audit_noise(s3, "Source 3")


==================== Source 1 ====================
Missing business_name: 0 (0.00%)
Missing business_address: 0 (0.00%)
Non-Latin names: 0 (0.00%)
Non-Latin addresses: 554 (0.03%)


C:\Users\Vibha Singh\AppData\Local\Temp\ipykernel_12020\786075851.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  domain = df['business_name'].str.contains(r'\.(com|in|org|net|co)\b|^@', regex=True, na=False)


Domain/handle names: 506 (0.02%)
'Formerly' mentions: 0 (0.00%)
All-caps names: 0 (0.00%)


C:\Users\Vibha Singh\AppData\Local\Temp\ipykernel_12020\786075851.py:32: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_suffix = df['business_name'].str.contains(suffix_pattern, case=False, regex=True, na=False)


Has legal suffix: 1378370 (62.46%)
Address has 5-6 digit code (PIN/ZIP): 149948 (6.79%)
Address length - mean: 52.1, median: 41, min: 11, max: 256
Country distribution:
country
US       1323633
India     883188
Name: count, dtype: int64

==================== Source 2 ====================
Missing business_name: 2 (0.00%)
Missing business_address: 168967 (3.36%)
Non-Latin names: 764608 (15.19%)
Non-Latin addresses: 478453 (9.50%)


...

Has legal suffix: 2438417 (48.43%)
Address has 5-6 digit code (PIN/ZIP): 389621 (7.74%)
Address length - mean: 47.8, median: 37, min: 8, max: 249
Country distribution:
country
US       3016817
India    2017799
Name: count, dtype: int64

==================== Source 3 ====================
Missing business_name: 13 (0.00%)
Missing business_address: 175916 (3.33%)
Non-Latin names: 606737 (11.48%)
Non-Latin addresses: 476588 (9.02%)


C:\Users\Vibha Singh\AppData\Local\Temp\ipykernel_12020\786075851.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  domain = df['business_name'].str.contains(r'\.(com|in|org|net|co)\b|^@', regex=True, na=False)


Domain/handle names: 184700 (3.49%)
'Formerly' mentions: 30057 (0.57%)
All-caps names: 156206 (2.96%)


C:\Users\Vibha Singh\AppData\Local\Temp\ipykernel_12020\786075851.py:32: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_suffix = df['business_name'].str.contains(suffix_pattern, case=False, regex=True, na=False)


Has legal suffix: 2687656 (50.85%)
Address has 5-6 digit code (PIN/ZIP): 406031 (7.68%)
Address length - mean: 48.3, median: 42, min: 2, max: 240
Country distribution:
country
US       3170056
India    2115547
Name: count, dtype: int64


In [4]:
import sys
print(sys.executable)

from unidecode import unidecode
print(unidecode("தமிழ்நாடு"))  # should print a transliterated ascii string

c:\Users\Vibha Singh\Desktop\amazon_ml\.venv\Scripts\python.exe
tmilllnaattu


In [5]:
from normalize import normalize_business_name, normalize_address, normalize_country

test_cases = [
    "Bnp Group of Companies-Delhi",
    "of Group Bnp Companies-Delhi Limited",
    "bnpgroupcompaniesdelhi.com",
    "Svl Tubes Pvt Ltd",
    "Irisyn One formerly: Tirth Resources Private Limited",
    "@jexfirst",
]

for t in test_cases:
    print(f"{t!r:55} -> {normalize_business_name(t)!r}")

'Bnp Group of Companies-Delhi'                          -> 'bnp group of companies delhi'
'of Group Bnp Companies-Delhi Limited'                  -> 'of group bnp companies delhi ltd'
'bnpgroupcompaniesdelhi.com'                            -> 'bnpgroupcompaniesdelhi'
'Svl Tubes Pvt Ltd'                                     -> 'svl tubes pvt ltd'
'Irisyn One formerly: Tirth Resources Private Limited'  -> 'tirth resources pvt ltd'
'@jexfirst'                                             -> 'jexfirst'


In [6]:
test_addresses = [
    "PLOT NO: ##51 FIRST FLOOR, GOVINDAN STREET, AYYAVOO COLONY, AMINJIK, ARAI, CHENNAI, தமிழ்நாடு",
    "920 Stinocher Saint, Soln, IA",
    "House No.-591, G.F, Bhai Parmanand Colony, Delhi, North West Delhi, Delhi",
]

for a in test_addresses:
    print(f"{a!r}\n  -> {normalize_address(a)!r}\n")

'PLOT NO: ##51 FIRST FLOOR, GOVINDAN STREET, AYYAVOO COLONY, AMINJIK, ARAI, CHENNAI, தமிழ்நாடு'
  -> 'plot no 51 first fl govindan st ayyavoo colony aminjik arai chennai tmilllnaattu'

'920 Stinocher Saint, Soln, IA'
  -> '920 stinocher st soln ia'

'House No.-591, G.F, Bhai Parmanand Colony, Delhi, North West Delhi, Delhi'
  -> 'house no 591 g f bhai parmanand colony delhi north west delhi delhi'



In [7]:
from tqdm import tqdm
tqdm.pandas()

s1['name_norm'] = s1['business_name'].progress_apply(normalize_business_name)
s1['addr_norm'] = s1['business_address'].progress_apply(normalize_address)

s2['name_norm'] = s2['business_name'].progress_apply(normalize_business_name)
s2['addr_norm'] = s2['business_address'].progress_apply(normalize_address)

s3['name_norm'] = s3['business_name'].progress_apply(normalize_business_name)
s3['addr_norm'] = s3['business_address'].progress_apply(normalize_address)

100%|██████████| 5285603/5285603 [01:45<00:00, 50253.94it/s]


In [9]:
s1.to_pickle("train/s1_normalized.pkl")
s2.to_pickle("train/s2_normalized.pkl")
s3.to_pickle("train/s3_normalized.pkl")

In [10]:
s1.to_csv("train/s1_normalized.tsv", sep="\t", index=False)
s2.to_csv("train/s2_normalized.tsv", sep="\t", index=False)
s3.to_csv("train/s3_normalized.tsv", sep="\t", index=False)

In [11]:
s1_slim = s1[['entity_id', 'name_norm', 'addr_norm', 'country']]
s2_slim = s2[['entity_id', 'name_norm', 'addr_norm', 'country']]
s3_slim = s3[['entity_id', 'name_norm', 'addr_norm', 'country']]

s1_slim.to_csv("train/s1_normalized_slim.tsv", sep="\t", index=False)
s2_slim.to_csv("train/s2_normalized_slim.tsv", sep="\t", index=False)
s3_slim.to_csv("train/s3_normalized_slim.tsv", sep="\t", index=False)

In [12]:
import os
for f in ["s1_normalized_slim.tsv", "s2_normalized_slim.tsv", "s3_normalized_slim.tsv"]:
    path = f"train/{f}"
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f, f"{size_mb:.1f} MB")

s1_normalized_slim.tsv 186.6 MB
s2_normalized_slim.tsv 411.8 MB
s3_normalized_slim.tsv 433.4 MB
